In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ಹಂತ 1: ರಚನೆಗೊಂಡ Outputs ಗಾಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟುಗಳು ಹಿಂತಿರುಗಿಸುವ **ಸ್ಕೀಮಾ** ಅನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ. ಪೈಡ್ಯಾಂಟಿಕ್ ನೊಂದಿಗೆ `response_format` ಬಳಕೆ ಅಗತ್ಯವಿದೆ:
- ✅ ಪ್ರಕಾರ-ಸುರಕ್ಷಿತ ಡೇಟಾ ಎಕ್ಸ್ಟ್ರ್ಯಾಕ್ಷನ್
- ✅ ಸ್ವಯಂಚಾಲಿತ ಪರಿಷ್ಕರಣೆ
- ✅ ಮುಕ್ತ-ಪಠ್ಯ ಉತ್ತರಗಳಿಂದ ಯಾವುದೇ ವಿಶ್ಲೇಷಣೆ ದೋಷಗಳಿಲ್ಲ
- ✅ ಕ್ಷೇತ್ರಗಳ ಆಧಾರದ ಮೇಲೆ ಸುಲಭ ಶರತ್ತು ಸಾರಿಯನ್ನು ನಿರ್ವಹಣೆ


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ಹಂತ 2: ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್ ಉಪಕರಣವನ್ನು ರಚಿಸಿ

ಈ ಉಪಕರಣವನ್ನು **availability_agent** ಮಂದಿರಗಳ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸಲು ಕರೆಮಾಡುತ್ತದೆ. ನಾವು `@ai_function` ಅಲಂಕರಣವನ್ನು ಬಳಸುತ್ತೇವೆ:
- ಪೈಥಾನ್ ಕಾರ್ಯವನ್ನು AI-ಕರೆ ಮಾಡಬಹುದಾದ ಉಪಕರಣಕ್ಕೆ ಪರಿವರ್ತಿಸಲು
- LLMಗಾಗಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ JSON ಸ್ಥಿತಿ ಪಟ್ಟಿ ರಚಿಸಲು
- ಪರಾಮಿತಿ ಮಾನ್ಯತೆ ನಿರ್ವಹಿಸಲು
- ಏಜೆಂಟ್ಸ್ ಮೂಲಕ ಸ್ವಯಂಚಾಲಿತ ಕರೆಗೆ ಸಾಧ್ಯವಾಗಿಸಲು

ಈ ಡೆಮೋಗಾಗಿ:
- **స్టాక్‌హోಮ್, సియాటిల్, టోక్యో, లండన్, ఆంస్టర్డ్యామ్** → ಗದ್ದೆಗಳು ಲಭ್ಯವಿದೆ ✅
- **ಮತ್ತೆ ಎಲ್ಲಾ ನಗರಗಳು** → ಗದ್ದೆಗಳು ಲಭ್ಯವಿಲ್ಲ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ಹಂತ 3: ಮಾರ್ಗ ನಿರ್ಧಾರಕ್ಕೆ ಸ್ಥಿತಿ ಕಾರ್ಯಗಳನ್ನು ನಿರ್ದಿಷ್ಟರಿಸಿ

ಈ ಕಾರ್ಯಗಳು ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸಿ ಕಾರ್ಯಪಟದಲ್ಲಿ ಯಾವ ದಾರಿಯನ್ನು ಕೈಗೊಳ್ಳಬೇಕೋ ನಿರ್ಧರಿಸುತ್ತವೆ.

**ಮುಖ್ಯ ಮಾದರಿ:**
1. ಸಂದೇಶವು `AgentExecutorResponse` ಆಗಿದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸಿ
2. ರಚನೆಗೊಂಡ ಕೊಡುಗೆ (ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿ) ಅನ್ನು ವಿದಾಯಿಸಿ
3. ಮಾರ್ಗ ನಿರ್ವಹಣೆಗೆ `True` ಅಥವಾ `False` ಅನ್ನು ಹಿಂತಿರುಗಿಸಿ

ಕಾರ್ಯಪಟವು ಈ ಸ್ಥಿತಿಗಳನ್ನು **ಮರಳುಹುಂದಿ**ಗಳಲ್ಲಿ ಮೌಲ್ಯಮಾಪನ ಮಾಡಿ ಮುಂದಿನ ಕಾರ್ಯನಿರ್ವಾಹಕರನ್ನು ಕರೆದೊಯ್ಯುವುದು ನಿರ್ಧರಿಸುತ್ತದೆ.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ಹಂತ 4: ಕಸ್ಟಮ್ ಡಿಸ್ಪ್ಲೇ Executor ರಚಿಸಿ

Executors ಅವು ವರ್ಕ್ಫ್ಲೋ ಘಟಕಗಳು, ಅವು ಪರಿವರ್ತನೆಗಳು ಅಥವಾ ಪಕ್ಕ ಪರಿಣಾಮಗಳನ್ನು ನಿರ್ವಹಿಸುತ್ತವೆ. ನಾವು ಅಂತಿಮ ಫಲಿತಾಂಶವನ್ನು ತೋರಿಸಲು `@executor` ಡೆಕೊರೇಟರ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ.

**ಪ್ರಮುಖ ಕಲ್ಪನೆಗಳು:**
- `@executor(id="...")` - ಕಾರ್ಯವನ್ನು ವರ್ಕ್ಫ್ಲೋ executor ಆಗಿ ನೋಂದಣಿ ಮಾಡುತ್ತದೆ
- `WorkflowContext[Never, str]` - ಇನ್ಪುಟ್/ಔಟ್‌ಪುಟ್ ಗಾಗಿ ಟೈಪ್ ಸೂಚನೆಗಳು
- `ctx.yield_output(...)` - ಅಂತಿಮ ವರ್ಕ್ಫ್ಲೋ ಫಲಿತಾಂಶವನ್ನು ನೀಡುತ್ತದೆ


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ಹಂತ 5: ಪಾರದರ್ಶಕತೆ ವ್ಯತ್ಯಾಸಗಳನ್ನು ಲೋಡ್ ಮಾಡಿ

LLM ಕ್ಲೈಯಂಟ್ ಅನ್ನು ಸಂರಚಿಸಿ. ಈ ಉದಾಹರಣೆಗಳೊಂದಿಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ:
- **Microsoft Foundry** / **Azure OpenAI** (ಪ್ರತಿಕ್ರಿಯೆಗಳು API — ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ಹೆಜ್ಜೆ 6: ರಚನಾತ್ಮಕ ಔಟ್‌ಪುಟ್‌ಗಳೊಂದಿಗೆ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸಿ

ನಾವು **ಮೂರು ವಿಶಿಷ್ಟ ಏಜೆಂಟ್‌ಗಳು** ರಚಿಸುತ್ತೇವೆ, ಪ್ರತಿಯೊಂದನ್ನು `AgentExecutor` ನಲ್ಲಿ ಒಳಗೊಂಡಿರುತ್ತದೆ:

1. **availability_agent** - ಉಪಕರಣ ಬಳಸಿ ಹೋಟೆಲ್ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುತ್ತದೆ
2. **alternative_agent** - ಪರ್ಯಾಯ ನಗರಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ (ಕೆಲವಾ ಕೊಠಡಿಗಳು ಇಲ್ಲದಿರುವಾಗ)
3. **booking_agent** - ಬುಕ್ಕಿಂಗ್ ಮಾಡಲು ಪ್ರೋತ್ಸಾಹಿಸುತ್ತದೆ (ಕೊಠಡಿಗಳು ಲಭ್ಯವಾಗಿದ್ದಾಗ)

**ಮುಖ್ಯ ವೈಶಿಷ್ಟ್ಯಗಳು:**
- `tools=[hotel_booking]` - ಏಜೆಂಟ್‌ಗೆ ಉಪಕರಣವನ್ನು ಒದಗಿಸುತ್ತದೆ
- `response_format=PydanticModel` - ರಚನಾತ್ಮಕ JSON ಔಟ್‌ಪುಟ್ ಅನ್ನು ಬಲವಂತ ಮಾಡುತ್ತದೆ
- `AgentExecutor(..., id="...")` - ಕೆಲಸಪ್ರವಾಹ ಬಳಕೆಗೆ ಏಜೆಂಟ್ ಅನ್ನು ಮಡಿಸುತ್ತದೆ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ಹಂತ 7: ಶರತ್ಗತ ಅಂಚುಗಳೊಂದಿಗೆ ವರ್ಕ್‌ಫ್ಲೋ ನಿರ್ಮಿಸಿ

ಈಗ ನಾವು `WorkflowBuilder` ಅನ್ನು ಶರತ್ಗತ ಮಾರ್ಗನಿರ್ದೇಶನದೊಂದಿಗೆ ಗ್ರಾಫ್ ನಿರ್ಮಿಸಲು ಬಳಸುತ್ತೇವೆ:

**ವರ್ಕ್‌ಫ್ಲೋ ರಚನೆ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ಪ್ರಮುಖ ವಿಧಾನಗಳು:**
- `.set_start_executor(...)` - ಪ್ರವೇಶ ಬಿಂದುವನ್ನು ಸೆಟ್ ಮಾಡುತ್ತದೆ
- `.add_edge(from, to, condition=...)` - ಶರತ್ಗತ ಅಂಚನ್ನು ಸೇರಿಸುತ್ತದೆ
- `.build()` - ವರ್ಕ್‌ಫ್ಲೋ ಅನ್ನು ಅಂತಿಮಗೊಳಿಸುತ್ತದೆ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ಹಂತ 8: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 1 ಅನ್ನು چلಾಯಿಸಿ - ಲಭ್ಯತೆ ರಹಿತ ನಗರ (ಪ್ಯಾರಿಸ್)

ಪ್ಯಾರಿಸ್‌ನಲ್ಲಿ (ನಮ್ಮ ಸಿಮ್ಯುಲೇಶನ್‌ನಲ್ಲಿ ಅಲ್ಲಿ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿಲ್ಲ) ಹೋಟೆಲ್‌ಗಳನ್ನು ಕೇಳುವುದರಿಂದ **ಲಭ್ಯತೆ ಇಲ್ಲ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ಹಂತ 9: ಟೆಸ್ಟ್ ಕೇಸ್ 2 ರನ್ ಮಾಡಿ - ಲಭ್ಯತೆ ಇರುವ ನಗರ (ಸ್ಟಾಕ್‌ಹೋಮ್)

ಈಗ ನಾವು **ಲಭ್ಯತೆ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ ಸ್ಟಾಕ್‌ಹೋಮ್‌ನಲ್ಲಿರುವ ಹೋಟೆಲ್ಗಳನ್ನು ವಿನಂತಿ ಮಾಡುವ ಮೂಲಕ (ಯಾವದಲ್ಲೂ ನಮ್ಮ ಅನುಕರಣೆಗಳಲ್ಲಿ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿದ್ದು).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ಮುಖ್ಯ ತತ್ವಗಳು ಮತ್ತು ಮುಂದಿನ ಹೆಜ್ಜೆಗಳು

### ✅ ನೀವು ಕಲಿತದ್ದು:

1. **WorkflowBuilder ಮಾದರಿ**
   - ಪ್ರಾರಂಭ ಬಿಂದುವನ್ನು ನಿಗದಿಪಡಿಸಲು `.set_start_executor()` ಉಪಯೋಗಿಸಿ
   - ಸ್ಥಿತಿಗಳು ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗಸೂಚಿಗಾಗಿ `.add_edge(from, to, condition=...)` ಬಳಸಿ
   - ವರ್ಕ್‌ಫ್ಲೋವನ್ನು ಅಂತಿಮಗೊಳಿಸಲು `.build()` ಕರೆಸಿರಿ

2. **ಸ್ಥಿತಿಗತ ಮಾರ್ಗಸೂಚಿ**
   - ಶರತ್ತು ಕಾರ್ಯಗಳು `AgentExecutorResponse` ಅನ್ನು ಪರಿಶೀಲಿಸುತ್ತವೆ
   - ಮಾರ್ಗ ನಿರ್ಣಯಕ್ಕಾಗಿ ರಚನಾತ್ಮಕ ಔಟ್ಪುಟ್‌ಗಳನ್ನು ಬ್ಯಾಖ್ಯಾನಿಸಿ
   - ಒಂದು ಎಡ್ಜ್ ಅನ್ನು ಸಕ್ರಿಯಗೊಳಿಸಲು `True` ಅನ್ನು ಹಿಂತಿರುಗಿಸಿ, ತಪ್ಪಿಸುವುದಕ್ಕೆ `False`

3. **ಟೂಲ್ ಸಂಯೋಜನೆ**
   - Python ಫಂಕ್ಷನ್‌ಗಳನ್ನು AI ಉಪಕರಣಗಳಾಗಿ ಪರಿವರ್ತಿಸಲು `@ai_function` ಬಳಸಿ
   - ಅಗತ್ಯವಿದ್ದಾಗ ಏಜೆಂಟ್‌ಗಳು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಉಪಕರಣಗಳನ್ನು ಕರೆಸುತ್ತವೆ
   - ಉಪಕರಣಗಳು JSON ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತವೆ, ಏಜೆಂಟ್ ಗಳು ಅದನ್ನು ವಿಶ್ಲೇಷಿಸುತ್ತವೆ

4. **ರಚನಾತ್ಮಕ ಔಟ್ಪುಟ್‌ಗಳು**
   - ಪ್ರಕಾರ-ಸುರಕ್ಷಿತ ಡೇಟಾ ಹೊರತೆಗೆದುಕೊಳ್ಳಲು Pydantic ಮಾದರಿಗಳನ್ನು ಬಳಸಿ
   - ಏಜೆಂಟ್‌ಗಳು ಸೃಷ್ಟಿಸುವಾಗ `response_format=MyModel` ಅನ್ನು ಸೆಟ್ ಮಾಡಿ
   - `Model.model_validate_json()` ಬಳಸಿ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ವಿಶ್ಲೇಷಿಸಿ

5. **ಕಸ್ಟಮ್ ಎಕ್ಸಿಕ್ಯೂಟರ್‌ಗಳು**
   - ವರ್ಕ್‌ಫ್ಲೋ ಘಟಕಗಳನ್ನು ಸೃಷ್ಟಿಸಲು `@executor(id="...")` ಬಳಸಿ
   - ಎಕ್ಸಿಕ್ಯೂಟರ್‌ಗಳು ಡೇಟಾವನ್ನು ಪರಿವರ್ತನೆ ಮಾಡಬಹುದು ಅಥವಾ ಬದಿಗುಚುವ ಪರಿಣಾಮಗಳನ್ನು ಮಾಡಬಹುದು
   - ವರ್ಕ್‌ಫ್ಲೋ ಫಲಿತಾಂಶಗಳನ್ನು ತಯಾರಿಸಲು `ctx.yield_output()` ಬಳಸಿ

### 🚀 ವಾಸ್ತವ ಜಾಗತಿಕ ಅನ್ವಯಿಕೆಗಳು:

- **ಪ್ರಯಾಣ ಬುಕಿಂಗ್**: ಲಭ್ಯತೆ ಪರಿಶೀಲನೆ, ಪರ್ಯಾಯ ಸಲಹೆ, ಆಯ್ಕೆಗಳು ಹೋಲಿಕೆ
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಸಮಸ್ಯೆಯ ಪ್ರಕಾರ, ಭಾವನೆ, ಆದ್ಯತೆ ಆಧಾರಿತ ಮಾರ್ಗಸೂಚಿ
- **ಇ-ಕಾಮರ್ಸ್**: ಇನ್ವೆಂಟರಿ ಪರಿಶೀಲನೆ, ಪರ್ಯಾಯ ಸಲಹೆ, ಆರ್ಡರ್ ಪ್ರాసೆಸ್
- **ವಿಷಯ ನಿಯಂತ್ರಣ**: ವಿಷಕಾರಿ ಅಂಕೆ, ಬಳಕೆದಾರ ಪಟ್ಟಿ ಆಧಾರಿತ ಮಾರ್ಗಸೂಚಿ
- **ಅನುವಾದ ಕಾರ್ಯಪ್ರವಾಹಗಳು**: ಮೊತ್ತ, ಬಳಕೆದಾರ ಪಾತ್ರ, ಅಪಾಯ ಮಟ್ಟ ಆಧಾರಿತ ಮಾರ್ಗಸೂಚಿ
- **ಬಹು-ಹಂತದ ಪ್ರಕ್ರಿಯೆ**: ಡೇಟಾ ಗುಣಮಟ್ಟ, ಪೂರ್ಣತೆ ಆಧಾರಿತ ಮಾರ್ಗಸೂಚಿ

### 📚 ಮುಂದಿನ ಹೆಜ್ಜೆಗಳು:

- ಹೆಚ್ಚಿನ ಸಂಕೀರ್ಣ ಶರತ್ತುಗಳನ್ನು ಸೇರಿಸಿ (ಬಹು ಮಾನದಂಡಗಳು)
- ವರ್ಕ್‌ಫ್ಲೋ ರಾಜ್ಯ ನಿರ್ವಹಣೆಯೊಂದಿಗೆ ಲೂಪ್ಸ್ ಅನ್ನು ಜಾರಿಗೆ ತಂದಿರಿ
- ಮರುಬಳಕೆ ಹೊಂದಬಹುದಾದ ಘಟಕಗಳಿಗಾಗಿ ಉಪ-ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ಸೇರಿಸಿ
- ವಾಸ್ತವ API ಗಳು (ಹೋಟೆಲ್ ಬುಕಿಂಗ್, ಇನ್ವೆಂಟರಿ ವ್ಯವಸ್ಥೆಗಳು) ಸಂಯೋಜಿಸಿ
- ದೋಷ ನಿರ್ವಹಣೆ ಮತ್ತು ಪತನ ಮಾರ್ಗಗಳನ್ನು ಸೇರಿಸಿ
- ಒಳಗೊಳ್ಳಲಾದ ದೃಶ್ಯೀಕರಣ ಉಪಕರಣಗಳೊಂದಿಗೆ ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ದೃಶ್ಯ ಮಾಡಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
